In [0]:
import requests
from requests.exceptions import RequestException
import pandas as pd
import time
import os
from pyspark.sql.functions import year, month, col, to_date
from datetime import timedelta

In [0]:
dbutils.widgets.text("bronze_catalog","dbr_dev")
dbutils.widgets.text("bronze_schema", "artemzharkov10_bronze")

dbutils.widgets.text("silver_catalog","dbr_dev")
dbutils.widgets.text("silver_schema","artemzharkov10_silver")

BRONZE_CATALOG = dbutils.widgets.get("bronze_catalog")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")

SILVER_CATALOG = dbutils.widgets.get("silver_catalog")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")

SOURCE_PATH = f"/Volumes/{BRONZE_CATALOG}/{BRONZE_SCHEMA}/raw_data/weather_history"

In [0]:
dbutils.fs.mkdirs(SOURCE_PATH)

In [0]:
grid_df = spark.sql(f"""
    SELECT 
        ROUND(gps_x, 2) AS lat, 
        ROUND(gps_y, 2) AS lon,
        CAST(accident_date AS DATE) AS acc_date
    FROM {SILVER_CATALOG}.{SILVER_SCHEMA}.silver_sewik_accidents
    WHERE gps_x IS NOT NULL AND gps_y IS NOT NULL AND accident_date IS NOT NULL AND accident_date > "2024-10-18"
    ORDER BY acc_date
""")
locations = grid_df.collect()

In [0]:
URL = "https://power.larc.nasa.gov/api/temporal/hourly/point"
NASA_HOURLY_PARAMS = "T2M,PRECTOTCORR,WS10M,TS,RH2M,T2MDEW"

# T2M	Температура воздуха на высоте 2 м	°C
# PRECTOTCORR	Скорректированное общее количество осадков	мм/день (для Daily) или мм/час (для Hourly)
# WS10M	Скорость ветра на высоте 10 м	м/с
# TS6 (у вас, вероятно, опечатка TSб)	Температура поверхности земли (Surface Skin Temperature)	°C
# RH2M	Относительная влажность на высоте 2 м	%
# T2MDEW	Температура точки росы на высоте 2 м	°C

In [0]:
total_locations = len(locations)
print(f"Всего уникальных событий для обработки: {total_locations}")

for index, row in enumerate(locations, 1):
    lat = row['lat']
    lon = row['lon']
    
    # Расчет дат для запроса (день аварии и предыдущий день)
    end_date_obj = row['acc_date'] 
    start_date_obj = end_date_obj - timedelta(days=1)
    
    # Формат даты, требуемый NASA (YYYYMMDD)
    start_date_str = start_date_obj.strftime('%Y%m%d')
    end_date_str = end_date_obj.strftime('%Y%m%d')
    
    # Дата для названия файла и колонки (стандартная)
    file_date_str = end_date_obj.strftime('%Y-%m-%d')
    file_path = f"{SOURCE_PATH}/weather_{lat}_{lon}_{file_date_str}.csv"
    

    # I do not use a table log for review present file, because I would be do insert in table each time when this file is created it is 160.000 additioanal inset operations. Is it ok ? Or how I should to do ? 
    if os.path.exists(file_path):
        print(f"[{index}/{total_locations}] Пропуск: Файл {file_path} уже существует.")
        continue
        
    api_params = {
        "parameters": NASA_HOURLY_PARAMS,
        "community": "AG",
        "longitude": lon,
        "latitude": lat,
        "start": start_date_str,
        "end": end_date_str,
        "format": "JSON"
    }
    
    max_retries = 3
    location_data = []
    
    for attempt in range(max_retries):
        try:
            response = requests.get(URL, params=api_params, timeout=(15, 30))
            
            if response.status_code == 200:
                data = response.json()
                parameters = data.get("properties", {}).get("parameter", {})
                
                if not parameters:
                    break
                    
                # Получаем все ключи-даты формата YYYYMMDDHH из первого параметра
                datetime_keys = list(parameters.get("T2M", {}).keys())
                
                for dt_key in datetime_keys:
                    hour_str = dt_key[-2:] # Извлекаем последние 2 символа (час)
                    hour_int = int(hour_str)
                    
                    # Фильтрация (0, 4, 8, 12, 16, 20)
                    if hour_int % 4 == 0:
                        location_data.append({
                            "lat": lat,
                            "lon": lon,
                            "target_acc_date": file_date_str,
                            "time": f"{dt_key[:4]}-{dt_key[4:6]}-{dt_key[6:8]} {hour_str}:00:00",
                            "temperature_2m": parameters.get("T2M", {}).get(dt_key),
                            "precipitation": parameters.get("PRECTOTCORR", {}).get(dt_key),
                            "wind_speed_10m": parameters.get("WS10M", {}).get(dt_key),
                            "soil_temp": parameters.get("TS", {}).get(dt_key),
                            "humidity_2m": parameters.get("RH2M", {}).get(dt_key),
                            "dew_point_2m": parameters.get("T2MDEW", {}).get(dt_key)
                        })
                
                if location_data:
                    df_filtered = pd.DataFrame(location_data)
                    # Очистка DataFrame от стандартных заглушек NASA (-999.0 -> NULL)
                    df_filtered = df_filtered.replace(-999.0, None)
                    
                    df_filtered.to_csv(file_path, index=False)
                    # print(f"[{index}/{total_locations}] Успех lat:{lat}, lon:{lon}, date:{file_date_str} | Записано строк: {len(df_filtered)}")
                    # print(f"[{index}/{total_locations} date:{file_date_str}]")
                
                break
                
            elif response.status_code == 429:
                print(f"[{index}/{total_locations}] Throttling NASA. Ожидание 30с. Попытка {attempt + 1}")
                time.sleep(30)
                continue 
                
            else:
                print(f"[{index}/{total_locations}] Ошибка API {response.status_code}. Причина: {response.text}")
                break 
                
        except RequestException as e:
            if attempt < max_retries - 1:
                time.sleep(5) 
            else:
                print(f"[{index}/{total_locations}] Сбой сети. Ошибка: {e}")
print("Все скачано")
    # Минимальная пауза, требуемая NASA POWER для предотвращения сброса соединения (throttling)


In [0]:
from pyspark.sql.types import StructType, StructField, DoubleType, StringType, TimestampType
from pyspark.sql.functions import year, to_date, col

# 1. Жестко задаем структуру файла NASA
nasa_schema = StructType([
    StructField("lat", DoubleType(), True),
    StructField("lon", DoubleType(), True),
    StructField("target_acc_date", StringType(), True),
    StructField("time", TimestampType(), True), 
    StructField("temperature_2m", DoubleType(), True),
    StructField("precipitation", DoubleType(), True),
    StructField("wind_speed_10m", DoubleType(), True),
    StructField("soil_temp", DoubleType(), True),
    StructField("humidity_2m", DoubleType(), True),
    StructField("dew_point_2m", DoubleType(), True)
])

# 2. Читаем файлы, отключив inferSchema и передав нашу схему
df_raw = spark.read.csv(
    f"{SOURCE_PATH}/weather_*.csv", 
    header=True, 
    schema=nasa_schema 
)

# 3. Фильтрация от мусора (опционально, убирает строки из старых файлов, где time = null)
df_clean = df_raw.filter(col("time").isNotNull())

TARGET_PATH = f"/Volumes/{BRONZE_CATALOG}/{BRONZE_SCHEMA}/raw_data/weather_year"

# 4. Трансформация и запись
df_clean.withColumn("year", year(to_date(col("time")))) \
      .write \
      .mode("overwrite") \
      .partitionBy("year") \
      .format("csv") \
      .option("header", "true") \
      .save(TARGET_PATH)

print(f"Данные успешно сгруппированы по годам и сохранены в {TARGET_PATH}")

In [0]:
import shutil

# Путь к целевой директории со всеми годами
source_dir = "/Volumes/dbr_dev/artemzharkov10_bronze/raw_data/weather_year"

# Путь и имя выходного файла (функция автоматически добавит расширение .zip)
output_filename = "/Volumes/dbr_dev/artemzharkov10_bronze/raw_data/weather_year_archive"

print(f"Запуск архивации директории {source_dir}...")

# Выполнение сжатия в формате zip
shutil.make_archive(output_filename, 'zip', source_dir)

print(f"Архивация завершена. Файл сохранен как: {output_filename}.zip")

In [0]:
# import shutil
# import os

# # 1. Задаем пути
# source_dir = "/Volumes/dbr_dev/artemzharkov10_bronze/raw_data/weather_year"
# tmp_archive_path = "/tmp/weather_year_archive"  # Временный путь на локальном диске кластера
# final_archive_path = "/Volumes/dbr_dev/artemzharkov10_bronze/raw_data/weather_year_archive.zip"

# print("Шаг 1: Создание архива на локальном диске кластера...")
# # Выполнение сжатия во временную папку (добавит .zip автоматически)
# shutil.make_archive(tmp_archive_path, 'zip', source_dir)
# print("Архив успешно сформирован.")

# print("Шаг 2: Копирование готового архива в Unity Catalog Volume...")
# # Копирование цельного файла в облачное хранилище
# shutil.copy(f"{tmp_archive_path}.zip", final_archive_path)

# print("Шаг 3: Очистка временных файлов...")
# # Удаление временного файла с диска кластера для освобождения места
# os.remove(f"{tmp_archive_path}.zip")

# print(f"Готово! Файл доступен для скачивания по пути: {final_archive_path}")

In [0]:
# # Имя таблицы в формате catalog.schema.table
# TABLE_NAME = f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.bronze_weather"

# df_clean.withColumn("year", year(to_date(col("time")))) \
#       .write \
#       .mode("overwrite") \
#       .partitionBy("year") \
#       .format("delta") \
#       .saveAsTable(TABLE_NAME)

# print(f"Таблица Delta успешно создана: {TABLE_NAME}")